# Feature Engineering for Attendance Dataset
Based on `Feature-engineering.txt`.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load the dataset
df = pd.read_csv('../data/attendance_dataset-V2.csv')

## 1. Drop or Transform

In [3]:
# Parse Date
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Week_Number: weeks since 22-Jun-2026
start_date = pd.to_datetime('22-06-2026', format='%d-%m-%Y')
df['Week_Number'] = ((df['Date'] - start_date).dt.days // 7) + 1

# Day_Number_of_Semester
df['Day_Number_of_Semester'] = (df['Date'] - start_date).dt.days + 1

# Drop Students_Present and Attendance_Percentage (Target variables)
df = df.drop(columns=['Students_Present', 'Attendance_Percentage'])

# We can also drop the original 'Date' column now
df = df.drop(columns=['Date'])

# Dropping constant columns that are not useful as features
df = df.drop(columns=['End_Time', 'Semester', 'Branch', 'Section', 'Classroom', 'Total_Enrolled'], errors='ignore')

## 2. Keep and Encode

In [4]:
le = LabelEncoder()

# Day_of_Week
day_map = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
df['Day_of_Week'] = df['Day_of_Week'].map(day_map)

# Start_Time: encode as hour (8, 9, 10, 11, 13 etc)
# e.g., '8.30 AM' -> 8, '1.30 PM' -> 13
def extract_hour(time_str):
    time_str = time_str.strip()
    parts = time_str.split(' ')
    time_part = parts[0]
    meridian = parts[1] if len(parts) > 1 else ''
    hour = int(time_part.split('.')[0])
    if meridian == 'PM' and hour != 12:
        hour += 12
    if meridian == 'AM' and hour == 12:
        hour = 0
    return hour

df['Start_Time'] = df['Start_Time'].apply(extract_hour)

# Subject and Faculty_ID
df['Subject'] = le.fit_transform(df['Subject'])
df['Faculty_ID'] = le.fit_transform(df['Faculty_ID'])

# Practical_Theory (Theory=0, Practical=1)
df['Practical_Theory'] = df['Practical_Theory'].map({'Theory': 0, 'Practical': 1})

# Binary encodes
binary_map = {'No': 0, 'Yes': 1}
df['Internal_Test_Week'] = df['Internal_Test_Week'].map(binary_map)
df['Assignment_Due'] = df['Assignment_Due'].map(binary_map)
df['Special_Event'] = df['Special_Event'].map(binary_map)

# Holiday_Before_After (No=0, Before=1, After=2)
holiday_map = {'No': 0, 'Before': 1, 'After': 2}
df['Holiday_Before_After'] = df['Holiday_Before_After'].map(holiday_map)

# Weather
df['Weather'] = le.fit_transform(df['Weather'])

# Gap_Since_Previous_Lecture (Same Day=0, 1 Day=1, 2 Days=2, etc)
def parse_gap(gap_str):
    if gap_str == 'Same Day':
        return 0
    else:
        return int(gap_str.split(' ')[0])

df['Gap_Since_Previous_Lecture'] = df['Gap_Since_Previous_Lecture'].apply(parse_gap)


## 3. Engineer New Features

In [8]:
# Is_First_Lecture_of_Day
df['Is_First_Lecture_of_Day'] = (df['Lecture_Number'] == 1).astype(int)

# Is_Afternoon (starts at 1.30 PM or later)
df['Is_Afternoon'] = (df['Start_Time'] >= 12).astype(int)

# Rolling_Avg_3: rolling average of the last 3 attendance values for that subject
df = df.sort_values(by=['Subject', 'Day_Number_of_Semester', 'Start_Time'])
df['Rolling_Avg_3'] = df.groupby('Subject')['Previous_Lecture_Attendance'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

# Fill any missing values resulting from rolling mean (if any)
df['Rolling_Avg_3'] = df['Rolling_Avg_3'].fillna(0)

# Reset index after sorting
df = df.reset_index(drop=True)

df.head()

,Day_of_Week,Lecture_Number,Start_Time,Subject,Faculty_ID,Previous_Lecture_Attendance,Gap_Since_Previous_Lecture,Practical_Theory,Internal_Test_Week,Assignment_Due,Holiday_Before_After,Weather,Special_Event,Week_Number,Day_Number_of_Semester,Is_First_Lecture_of_Day,Is_Afternoon,Rolling_Avg_3
0,1,5,13,0,4,39,2,1,1,0,0,0,0,1,2,0,1,39.000000
1,1,5,13,0,4,79,7,1,1,0,0,0,1,2,9,0,1,59.000000
2,1,5,13,0,4,16,1,1,0,1,1,2,0,3,16,0,1,44.666667
3,1,5,13,0,4,13,2,1,0,1,0,0,0,4,23,0,1,36.000000
4,1,5,13,0,4,15,2,1,0,0,0,2,1,5,30,0,1,14.666667
